In [ ]:
# ============================================================
# SegFormer Fine-Tuning: Minecraft -> Cityscapes
# ============================================================

# === CELL 1: Setup ===
!pip install -q transformers==4.44.2 "albumentations==1.4.14" "albucore==0.0.16" accelerate evaluate

import os, random, json, glob, zipfile
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

# Class scheme
NUM_CLASSES = 4
CLASS_NAMES = ["road", "structure", "vegetation", "sky"]
IGNORE_INDEX = 255

# Palette for visualization (matches your scheme)
PALETTE = np.array([
    [255, 0, 0],     # 0 road
    [0, 0, 255],     # 1 building
    [0, 255, 0],     # 2 vegetation
    [135, 206, 235], # 3 sky (sky-blue for viz; actual capture uses gradient)
], dtype=np.uint8)

In [ ]:
# === CELL 2: Mount Drive and unzip dataset ===
from google.colab import drive
drive.mount('/content/drive')

# EDIT THIS PATH to wherever you put the zip
ZIP_PATH = "PATH_TO_DATA_ZIP""
DATA_ROOT = "/content/data"

os.makedirs(DATA_ROOT, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(DATA_ROOT)

# Expecting: DATA_ROOT/rgb/*.png and DATA_ROOT/mask_label/*.png
def find_dir(name):
    matches = [p for p in glob.glob(f"{DATA_ROOT}/**/{name}", recursive=True)
               if "__MACOSX" not in p]
    return sorted(matches)[0]

RGB_DIR = find_dir("rgb")
MASK_DIR = find_dir("mask_label")
print("RGB dir:", RGB_DIR)
print("Mask dir:", MASK_DIR)
print("RGB count:", len([f for f in os.listdir(RGB_DIR) if f.endswith('.png')]))
print("Mask count:", len([f for f in os.listdir(MASK_DIR) if f.endswith('.png')]))

In [ ]:
# === CELL 3: Train/val split ===
rgb_files = sorted([f for f in os.listdir(RGB_DIR) if f.endswith('.png')])
# Pair by filename — assumes rgb_XXXX.png <-> mask_XXXX.png with same number
def mask_name_from_rgb(rgb_name):
    # rgb_0001.png -> mask_0001.png
    return rgb_name.replace("rgb_", "mask_")

pairs = []
for rf in rgb_files:
    mf = mask_name_from_rgb(rf)
    if os.path.exists(os.path.join(MASK_DIR, mf)):
        pairs.append((rf, mf))
print(f"Paired {len(pairs)} / {len(rgb_files)} RGB files")

random.Random(SEED).shuffle(pairs)
N_VAL = 30
val_pairs = pairs[:N_VAL]
train_pairs = pairs[N_VAL:]
print(f"Train: {len(train_pairs)} | Val: {len(val_pairs)}")

In [ ]:
# === CELL 4: Augmentations ===
IMG_SIZE = 512  # square crop for training

# ImageNet stats (SegFormer expects these)
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

train_tf = A.Compose([
    A.RandomScale(scale_limit=(-0.5, 1.0), p=1.0),  # 0.5x to 2.0x
    A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE,
                  border_mode=0, value=0, mask_value=IGNORE_INDEX),
    A.RandomCrop(height=IMG_SIZE, width=IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    # Aggressive color jitter — implicit domain randomization (Tobin et al. flavor)
    A.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1, p=0.8),
    A.RandomGamma(gamma_limit=(70, 130), p=0.3),
    A.GaussianBlur(blur_limit=(3, 7), p=0.2),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

# Val: just resize + normalize, keep aspect ratio with padding
val_tf = A.Compose([
    A.LongestMaxSize(max_size=IMG_SIZE),
    A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE,
                  border_mode=0, value=0, mask_value=IGNORE_INDEX),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

In [ ]:
# === CELL 5: Dataset ===
class MinecraftSegDataset(Dataset):
    def __init__(self, pairs, rgb_dir, mask_dir, transform):
        self.pairs = pairs
        self.rgb_dir = rgb_dir
        self.mask_dir = mask_dir
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        rf, mf = self.pairs[idx]
        img = np.array(Image.open(os.path.join(self.rgb_dir, rf)).convert("RGB"))
        mask = np.array(Image.open(os.path.join(self.mask_dir, mf)))
        if mask.ndim == 3:
            mask = mask[..., 0]  # in case mask is saved as RGB
        out = self.transform(image=img, mask=mask)
        return out["image"], out["mask"].long()

train_ds = MinecraftSegDataset(train_pairs, RGB_DIR, MASK_DIR, train_tf)
val_ds = MinecraftSegDataset(val_pairs, RGB_DIR, MASK_DIR, val_tf)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False,
                        num_workers=2, pin_memory=True)

# Sanity check one batch
xb, yb = next(iter(train_loader))
print("Image batch:", xb.shape, xb.dtype, "min/max:", xb.min().item(), xb.max().item())
print("Mask batch:", yb.shape, yb.dtype, "unique labels:", torch.unique(yb).tolist())

In [ ]:
# === CELL 6: Visualize a few train samples (sanity check augmentation) ===
def denorm(x):
    mean = torch.tensor(MEAN).view(3,1,1)
    std = torch.tensor(STD).view(3,1,1)
    return (x.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

def colorize(mask):
    out = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for i, c in enumerate(PALETTE):
        out[mask == i] = c
    out[mask == IGNORE_INDEX] = [0, 0, 0]
    return out

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for i in range(4):
    img, msk = train_ds[i]
    axes[0, i].imshow(denorm(img)); axes[0, i].axis('off'); axes[0, i].set_title(f"img {i}")
    axes[1, i].imshow(colorize(msk.numpy())); axes[1, i].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# === CELL 7: Model ===
MODEL_NAME = "nvidia/segformer-b2-finetuned-ade-512-512"

model = SegformerForSemanticSegmentation.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
    id2label={i: n for i, n in enumerate(CLASS_NAMES)},
    label2id={n: i for i, n in enumerate(CLASS_NAMES)},
    ignore_mismatched_sizes=True,  # replaces classification head
)
model.to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {n_params/1e6:.1f}M")

In [ ]:
# === CELL 8: Optimizer & scheduler ===
EPOCHS = 40
LR_BACKBONE = 6e-6
LR_HEAD = 6e-5
WD = 0.01

# Separate param groups for backbone vs head
backbone_params, head_params = [], []
for name, p in model.named_parameters():
    if not p.requires_grad: continue
    if "decode_head" in name:
        head_params.append(p)
    else:
        backbone_params.append(p)

optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": LR_BACKBONE},
    {"params": head_params, "lr": LR_HEAD},
], weight_decay=WD)

steps_per_epoch = len(train_loader)
total_steps = EPOCHS * steps_per_epoch

# Polynomial decay (SegFormer convention, power 1.0 = linear)
def poly_lr(step, total, power=1.0, min_factor=0.0):
    return max(min_factor, (1 - step / total) ** power)

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda s: poly_lr(s, total_steps))

In [ ]:
# === CELL 9: Metrics ===
def fast_hist(pred, target, n):
    """Build confusion matrix on GPU for IoU computation."""
    mask = (target >= 0) & (target < n)
    return torch.bincount(n * target[mask].long() + pred[mask].long(),
                          minlength=n*n).reshape(n, n)

def iou_from_hist(hist):
    intersection = torch.diag(hist).float()
    union = hist.sum(0).float() + hist.sum(1).float() - intersection
    iou = intersection / union.clamp(min=1)
    return iou

In [ ]:
# === CELL 10: Train + eval loop ===
@torch.no_grad()
CKPT_DIR = "PATH_TO_CHECKPOINT_DIR"
os.makedirs(CKPT_DIR, exist_ok=True)

def evaluate(model, loader):
    model.eval()
    hist = torch.zeros(NUM_CLASSES, NUM_CLASSES, device=device)
    losses = []
    for img, msk in loader:
        img, msk = img.to(device), msk.to(device)
        out = model(pixel_values=img, labels=msk)
        losses.append(out.loss.item())
        # Logits are at H/4, W/4 — upsample to mask size
        logits = F.interpolate(out.logits, size=msk.shape[-2:],
                               mode="bilinear", align_corners=False)
        pred = logits.argmax(1)
        valid = msk != IGNORE_INDEX
        hist += fast_hist(pred[valid], msk[valid], NUM_CLASSES)
    iou = iou_from_hist(hist)
    return {"loss": float(np.mean(losses)),
            "miou": iou.mean().item(),
            "per_class_iou": iou.cpu().tolist()}

history = []
best_miou = 0.0
global_step = 0

for epoch in range(EPOCHS):
    model.train()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    epoch_losses = []
    for img, msk in pbar:
        img, msk = img.to(device), msk.to(device)
        out = model(pixel_values=img, labels=msk)
        loss = out.loss
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        epoch_losses.append(loss.item())
        global_step += 1
        pbar.set_postfix({"loss": f"{loss.item():.3f}",
                          "lr_h": f"{optimizer.param_groups[1]['lr']:.2e}"})

    val_metrics = evaluate(model, val_loader)
    train_loss = float(np.mean(epoch_losses))
    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f} | "
          f"val_loss={val_metrics['loss']:.4f} | val_mIoU={val_metrics['miou']:.4f} | "
          f"per_class={[f'{x:.2f}' for x in val_metrics['per_class_iou']]}")
    history.append({"epoch": epoch+1, "train_loss": train_loss, **val_metrics})

    if val_metrics["miou"] > best_miou:
        best_miou = val_metrics["miou"]
        torch.save({"model": model.state_dict(),
                    "epoch": epoch+1,
                    "miou": best_miou},
                   f"{CKPT_DIR}/best.pt")
        print(f"  -> new best, saved (mIoU={best_miou:.4f})")

# Save history + final
with open(f"{CKPT_DIR}/history.json", "w") as f:
    json.dump(history, f, indent=2)
torch.save(model.state_dict(), f"{CKPT_DIR}/final.pt")
print(f"Done. Best Minecraft-val mIoU: {best_miou:.4f}")

In [ ]:
# === CELL 11: Plot training curves ===
import matplotlib.pyplot as plt
epochs = [h["epoch"] for h in history]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(epochs, [h["train_loss"] for h in history], label="train")
ax[0].plot(epochs, [h["loss"] for h in history], label="val")
ax[0].set_title("Loss"); ax[0].legend(); ax[0].set_xlabel("epoch")
ax[1].plot(epochs, [h["miou"] for h in history])
ax[1].set_title("Val mIoU"); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.show()

In [ ]:
# Better viz: undo the letterbox padding so we compare like-for-like
# val_tf does LongestMaxSize(512) then PadIfNeeded to 512x512, so pad
# is symmetric on top/bottom (since input is wider than tall)
@torch.no_grad()
def predict_one(img_tensor):
    img_tensor = img_tensor.unsqueeze(0).to(device)
    out = model(pixel_values=img_tensor)
    logits = F.interpolate(out.logits, size=img_tensor.shape[-2:],
                           mode="bilinear", align_corners=False)
    return logits.argmax(1)[0].cpu().numpy()

def crop_to_content(img, mask, pred):
    """Find rows where mask != IGNORE_INDEX and crop all three to that range."""
    valid_rows = (mask != IGNORE_INDEX).any(axis=1)
    if not valid_rows.any():
        return img, mask, pred
    r0, r1 = np.where(valid_rows)[0][[0, -1]]
    return img[r0:r1+1], mask[r0:r1+1], pred[r0:r1+1]

N_SHOW = 6
indices = np.linspace(0, len(val_ds)-1, N_SHOW, dtype=int)
fig, axes = plt.subplots(N_SHOW, 3, figsize=(15, 3*N_SHOW))
for row, idx in enumerate(indices):
    img_t, msk_t = val_ds[idx]
    pred = predict_one(img_t)
    img_np = denorm(img_t)
    msk_np = msk_t.numpy()
    img_np, msk_np, pred = crop_to_content(img_np, msk_np, pred)
    axes[row, 0].imshow(img_np); axes[row, 0].set_title(f"Minecraft RGB (val {idx})")
    axes[row, 1].imshow(colorize(msk_np)); axes[row, 1].set_title("GT")
    axes[row, 2].imshow(colorize(pred)); axes[row, 2].set_title("Prediction")
    for ax in axes[row]: ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# === CELL 12: Unzip Cityscapes val ===
CITYSCAPES_ZIP = "/content/drive/MyDrive/cv_final_project/cityscapes_val.zip"
CITYSCAPES_ROOT = "/content/cityscapes"

import os, zipfile, glob
os.makedirs(CITYSCAPES_ROOT, exist_ok=True)
with zipfile.ZipFile(CITYSCAPES_ZIP) as z:
    z.extractall(CITYSCAPES_ROOT)

# Find image and label dirs (filter out __MACOSX as before)
def find_dir(name, root):
    matches = [p for p in glob.glob(f"{root}/**/{name}", recursive=True)
               if "__MACOSX" not in p]
    return sorted(matches)[0] if matches else None

CITY_IMG_DIR = find_dir("val", f"{CITYSCAPES_ROOT}/**/leftImg8bit")
CITY_LBL_DIR = find_dir("val", f"{CITYSCAPES_ROOT}/**/gtFine")
print("Cityscapes images:", CITY_IMG_DIR)
print("Cityscapes labels:", CITY_LBL_DIR)

# Cityscapes val has city subfolders: frankfurt, lindau, munster
img_files = sorted(glob.glob(f"{CITY_IMG_DIR}/**/*_leftImg8bit.png", recursive=True))
print(f"Found {len(img_files)} val images (expect 500)")

In [ ]:
# === CELL 13: Cityscapes -> 4-class remap LUT ===
# Build a 256-entry lookup table: cityscapes labelId -> our class
# (labelIds.png contains values 0-33, plus 255 for void)
import numpy as np

# Our 4-class scheme:
# 0 = road, 1 = structure, 2 = vegetation, 3 = sky, 255 = ignore
CITY_TO_OURS = np.full(256, 255, dtype=np.uint8)  # default: ignore
CITY_TO_OURS[7]  = 0  # road -> road
CITY_TO_OURS[8]  = 1  # sidewalk -> structure
CITY_TO_OURS[11] = 1  # building -> structure
CITY_TO_OURS[12] = 1  # wall -> structure
CITY_TO_OURS[13] = 1  # fence -> structure
CITY_TO_OURS[21] = 2  # vegetation -> vegetation
# 22 (terrain) -> 255 (ignore)  [explicitly: not vegetation]
CITY_TO_OURS[23] = 3  # sky -> sky
# Everything else (people 24, rider 25, car 26, truck 27, bus 28,
# train 31, motorcycle 32, bicycle 33, pole 17, traffic light 19,
# traffic sign 20, etc.) stays at 255 (ignore)

# Sanity check
print("Mapping (cityscapes_id -> our_class):")
for cid in [7, 8, 11, 12, 13, 21, 22, 23, 24, 26]:
    name = {7:"road", 8:"sidewalk", 11:"building", 12:"wall", 13:"fence",
            21:"vegetation", 22:"terrain", 23:"sky", 24:"person", 26:"car"}[cid]
    print(f"  {cid:3d} ({name:12s}) -> {CITY_TO_OURS[cid]}")

In [ ]:
# === CELL 14: Cityscapes label name from image name ===
def label_path_from_image(img_path):
    """
    leftImg8bit/val/frankfurt/frankfurt_000000_000294_leftImg8bit.png
       -> gtFine/val/frankfurt/frankfurt_000000_000294_gtFine_labelIds.png
    """
    rel = img_path.replace(CITY_IMG_DIR, "").lstrip("/")
    rel = rel.replace("_leftImg8bit.png", "_gtFine_labelIds.png")
    return os.path.join(CITY_LBL_DIR, rel)

# Verify pairing
test_img = img_files[0]
test_lbl = label_path_from_image(test_img)
print("Image:", test_img)
print("Label:", test_lbl)
print("Label exists:", os.path.exists(test_lbl))

In [ ]:
# === CELL 15: Cityscapes dataset (eval-only, no augmentation) ===
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Eval at 1024x512 (downsized from native 2048x1024)
EVAL_W, EVAL_H = 1024, 512

city_eval_tf = A.Compose([
    A.Resize(height=EVAL_H, width=EVAL_W, interpolation=1),  # bilinear for image
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

class CityscapesValDataset(Dataset):
    def __init__(self, img_files, transform, lut):
        self.img_files = img_files
        self.transform = transform
        self.lut = lut

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_path = self.img_files[idx]
        lbl_path = label_path_from_image(img_path)
        img = np.array(Image.open(img_path).convert("RGB"))
        lbl_raw = np.array(Image.open(lbl_path))
        # Remap before resize so we can use NEAREST on remapped labels
        lbl = self.lut[lbl_raw]
        # Albumentations resize w/ mask uses nearest neighbor for mask by default
        # when we pass it via the `mask` argument
        out = self.transform(image=img, mask=lbl)
        return out["image"], out["mask"].long(), img_path

city_ds = CityscapesValDataset(img_files, city_eval_tf, CITY_TO_OURS)
city_loader = DataLoader(city_ds, batch_size=4, shuffle=False,
                         num_workers=2, pin_memory=True)

# Sanity check
xb, yb, paths = next(iter(city_loader))
print("Batch image shape:", xb.shape)
print("Batch mask shape:", yb.shape, "unique:", torch.unique(yb).tolist())
# Expect classes {0, 1, 2, 3, 255}

In [ ]:
# === CELL 16: Reload model from scratch ===
# Build a fresh model and load best.pt — guarantees no stale state
import torch
from transformers import SegformerForSemanticSegmentation

CKPT_DIR = "/content/drive/MyDrive/cv_final_project/checkpoints"
ckpt_path = f"{CKPT_DIR}/best.pt"
print(f"Loading checkpoint: {ckpt_path}")
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
print(f"  Checkpoint epoch: {ckpt['epoch']}")
print(f"  Checkpoint Minecraft-val mIoU: {ckpt['miou']:.4f}")

# Rebuild model architecture
model = SegformerForSemanticSegmentation.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
    id2label={i: n for i, n in enumerate(CLASS_NAMES)},
    label2id={n: i for i, n in enumerate(CLASS_NAMES)},
    ignore_mismatched_sizes=True,
)

# Load saved weights
missing, unexpected = model.load_state_dict(ckpt["model"], strict=True)
print(f"  Missing keys: {len(missing) if missing else 0}")
print(f"  Unexpected keys: {len(unexpected) if unexpected else 0}")

model.to(device)
model.eval()
print("Model loaded and in eval mode.")

In [ ]:
# === CELL 17: Run Cityscapes evaluation ===
import torch.nn.functional as F
from tqdm.auto import tqdm

@torch.no_grad()
def evaluate_cityscapes(model, loader):
    model.eval()
    hist = torch.zeros(NUM_CLASSES, NUM_CLASSES, device=device, dtype=torch.float64)
    for img, msk, _ in tqdm(loader, desc="Cityscapes eval"):
        img, msk = img.to(device), msk.to(device)
        out = model(pixel_values=img)
        logits = F.interpolate(out.logits, size=msk.shape[-2:],
                               mode="bilinear", align_corners=False)
        pred = logits.argmax(1)
        valid = msk != IGNORE_INDEX
        if valid.sum() == 0:
            continue
        hist += fast_hist(pred[valid], msk[valid], NUM_CLASSES).double()
    iou = iou_from_hist(hist)
    # Per-class accuracy too (diag / row sum = recall-ish)
    acc_per_class = torch.diag(hist) / hist.sum(1).clamp(min=1)
    return {
        "miou": iou.mean().item(),
        "per_class_iou": iou.cpu().tolist(),
        "per_class_acc": acc_per_class.cpu().tolist(),
        "hist": hist.cpu().numpy(),
    }

city_results = evaluate_cityscapes(model, city_loader)
print("\n=== Cityscapes val results (4-class) ===")
print(f"mIoU: {city_results['miou']:.4f}")
print(f"{'class':<12} {'IoU':>8} {'Acc':>8}")
for i, name in enumerate(CLASS_NAMES):
    print(f"{name:<12} {city_results['per_class_iou'][i]:>8.4f} "
          f"{city_results['per_class_acc'][i]:>8.4f}")

In [ ]:
# === CELL 18: Save results + confusion matrix ===
import json
results_to_save = {
    "miou": city_results["miou"],
    "per_class_iou": city_results["per_class_iou"],
    "per_class_acc": city_results["per_class_acc"],
    "class_names": CLASS_NAMES,
    "eval_resolution": [EVAL_W, EVAL_H],
    "n_images": len(img_files),
}
with open(f"{CKPT_DIR}/cityscapes_results.json", "w") as f:
    json.dump(results_to_save, f, indent=2)

# Confusion matrix (normalized by row = recall)
import matplotlib.pyplot as plt
hist = city_results["hist"]
cm_norm = hist / hist.sum(axis=1, keepdims=True).clip(min=1)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Predicted"); ax.set_ylabel("Ground truth")
ax.set_title("Cityscapes confusion matrix (row-normalized)")
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, f"{cm_norm[i,j]:.2f}", ha="center", va="center",
                color="white" if cm_norm[i,j] > 0.5 else "black", fontsize=10)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(f"{CKPT_DIR}/cityscapes_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# === CELL 19: Qualitative samples ===
# Save side-by-side: RGB | GT | prediction for a handful of images
def colorize(mask):
    out = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for i, c in enumerate(PALETTE):
        out[mask == i] = c
    out[mask == IGNORE_INDEX] = [80, 80, 80]  # gray for ignore
    return out

@torch.no_grad()
def predict_one(img_tensor):
    img_tensor = img_tensor.unsqueeze(0).to(device)
    out = model(pixel_values=img_tensor)
    logits = F.interpolate(out.logits, size=img_tensor.shape[-2:],
                           mode="bilinear", align_corners=False)
    return logits.argmax(1)[0].cpu().numpy()

model.eval()
N_SHOW = 6
indices = np.linspace(0, len(city_ds)-1, N_SHOW, dtype=int)

fig, axes = plt.subplots(N_SHOW, 3, figsize=(15, 4*N_SHOW))
for row, idx in enumerate(indices):
    img, msk, path = city_ds[idx]
    pred = predict_one(img)
    img_np = denorm(img)
    axes[row, 0].imshow(img_np); axes[row, 0].set_title(f"RGB ({os.path.basename(path)[:30]}...)")
    axes[row, 1].imshow(colorize(msk.numpy())); axes[row, 1].set_title("Ground truth (4-class)")
    axes[row, 2].imshow(colorize(pred)); axes[row, 2].set_title("Prediction")
    for ax in axes[row]: ax.axis("off")
plt.tight_layout()
plt.savefig(f"{CKPT_DIR}/cityscapes_qualitative.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved qualitative samples to {CKPT_DIR}/cityscapes_qualitative.png")

In [ ]:
# === CELL 20: Load ADE20K-pretrained model with original 150-class head ===
# Note: this is a SEPARATE model instance from the fine-tuned one.
# We keep the 150-class head intact and remap predictions afterward.
import torch
from transformers import SegformerForSemanticSegmentation

baseline_model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b2-finetuned-ade-512-512"
).to(device)
baseline_model.eval()
print("Baseline model loaded (ADE20K-pretrained, 150 classes)")
print(f"  Output classes: {baseline_model.config.num_labels}")

In [ ]:
# === CELL 21: ADE20K -> 4-class remap ===
# ADE20K has 150 classes (1-indexed in their docs, 0-indexed in model output).
# Reference: https://github.com/CSAILVision/ADE20K
# We map ADE class index -> our class. Default = 255 (ignore).
#
# Key ADE classes (model outputs 0-indexed, so subtract 1 from "official" IDs):
#   0  = wall            -> structure
#   1  = building        -> structure
#   2  = sky             -> sky
#   3  = floor           -> 255 (indoor, irrelevant)
#   4  = tree            -> vegetation
#   5  = ceiling         -> 255
#   6  = road            -> road
#   8  = sidewalk        -> structure  (consistent with our Cityscapes mapping)
#   9  = earth/ground    -> 255 (ambiguous: could be road in dirt scenes, vegetation
#                                in parks, neither in interiors. Safest = ignore)
#   13 = grass           -> vegetation
#   16 = mountain        -> 255 (not in urban scenes; could be vegetation or ignore)
#   17 = plant           -> vegetation
#   25 = house           -> structure
#   29 = field           -> vegetation
#   32 = sand            -> 255
#   46 = palm tree       -> vegetation
#   52 = path            -> 255 (ambiguous, could be road or sidewalk)
#   53 = stairs          -> structure
#   59 = stairway        -> structure
#   68 = hill            -> 255
#   72 = land            -> 255
#   84 = fence           -> structure
#   90 = bridge          -> structure
#   94 = pole            -> 255 (Cityscapes also ignores poles for us)
#   96 = bannister       -> structure
#   100 = runway         -> road
#   123 = trade name     -> 255
#   136 = traffic light  -> 255
#
# Everything else (people, vehicles, indoor objects, etc.) -> 255

import numpy as np
ADE_TO_OURS = np.full(150, 255, dtype=np.uint8)

# Structure (buildings, walls, urban verticals + sidewalks)
for idx in [0, 1, 25, 53, 59, 84, 90, 96]:
    ADE_TO_OURS[idx] = 1
ADE_TO_OURS[8] = 1  # sidewalk -> structure (matches our Cityscapes mapping)

# Sky
ADE_TO_OURS[2] = 3

# Road
ADE_TO_OURS[6] = 0
ADE_TO_OURS[100] = 0  # runway

# Vegetation
for idx in [4, 13, 17, 29, 46]:
    ADE_TO_OURS[idx] = 2

# Sanity print
mapped_to = {0: "road", 1: "structure", 2: "vegetation", 3: "sky", 255: "ignore"}
n_per = {v: int((ADE_TO_OURS == v).sum()) for v in [0, 1, 2, 3, 255]}
print("ADE class count per mapped target:")
for k, name in mapped_to.items():
    print(f"  {name:10s}: {n_per[k]} ADE classes")

In [ ]:
# === CELL 22: Evaluate baseline on Cityscapes ===
import torch.nn.functional as F
from tqdm.auto import tqdm

@torch.no_grad()
def evaluate_baseline_cityscapes(model_150, loader, ade_lut):
    """
    model_150 outputs 150 ADE classes. We argmax to get an ADE class per pixel,
    then remap to our 4 classes via the LUT.
    """
    model_150.eval()
    hist = torch.zeros(NUM_CLASSES, NUM_CLASSES, device=device, dtype=torch.float64)
    ade_lut_t = torch.from_numpy(ade_lut.astype(np.int64)).to(device)
    for img, msk, _ in tqdm(loader, desc="Baseline eval"):
        img, msk = img.to(device), msk.to(device)
        out = model_150(pixel_values=img)
        logits = F.interpolate(out.logits, size=msk.shape[-2:],
                               mode="bilinear", align_corners=False)
        ade_pred = logits.argmax(1)  # values in [0, 149]
        # Remap ADE prediction -> our class
        pred = ade_lut_t[ade_pred]   # values in {0, 1, 2, 3, 255}
        # Pixels predicted as 255 (ignore) need to be counted somewhere — they're
        # "wrong" by definition since GT only has classes 0-3 + ignore.
        # Standard practice: treat them as a misclassification; we count them
        # against the model by mapping 255 -> a sentinel that won't match any GT class.
        # Since fast_hist already filters target>=0 & target<n, predictions outside
        # [0,n) will NOT contribute to hist — they effectively vanish.
        # That artificially inflates IoU. Fix: clip 255 -> 0 (any class) so they
        # count as wrong against whatever the GT is.
        # Better approach: leave them as 255 and exclude from BOTH pred and GT
        # (which is what we do below). This measures IoU only on pixels where
        # baseline made a "valid" prediction. We'll also report coverage.
        valid = (msk != IGNORE_INDEX) & (pred != 255)
        if valid.sum() == 0:
            continue
        hist += fast_hist(pred[valid], msk[valid], NUM_CLASSES).double()
    iou = iou_from_hist(hist)
    acc_per_class = torch.diag(hist) / hist.sum(1).clamp(min=1)
    return {
        "miou": iou.mean().item(),
        "per_class_iou": iou.cpu().tolist(),
        "per_class_acc": acc_per_class.cpu().tolist(),
        "hist": hist.cpu().numpy(),
    }

baseline_results = evaluate_baseline_cityscapes(baseline_model, city_loader, ADE_TO_OURS)
print("\n=== Baseline (ADE20K zero-shot) Cityscapes results ===")
print(f"mIoU: {baseline_results['miou']:.4f}")
print(f"{'class':<12} {'IoU':>8} {'Acc':>8}")
for i, name in enumerate(CLASS_NAMES):
    print(f"{name:<12} {baseline_results['per_class_iou'][i]:>8.4f} "
          f"{baseline_results['per_class_acc'][i]:>8.4f}")

In [ ]:
# === CELL 23: Coverage diagnostic for baseline ===
# How much of the valid (non-ignore) GT did the baseline predict as "something
# in our 4 classes" vs "ignore (no ADE class maps here)"?
@torch.no_grad()
def compute_baseline_coverage(model_150, loader, ade_lut):
    model_150.eval()
    ade_lut_t = torch.from_numpy(ade_lut.astype(np.int64)).to(device)
    valid_total = 0
    covered_total = 0
    for img, msk, _ in loader:
        img, msk = img.to(device), msk.to(device)
        out = model_150(pixel_values=img)
        logits = F.interpolate(out.logits, size=msk.shape[-2:],
                               mode="bilinear", align_corners=False)
        ade_pred = logits.argmax(1)
        pred = ade_lut_t[ade_pred]
        valid = msk != IGNORE_INDEX
        covered = valid & (pred != 255)
        valid_total += valid.sum().item()
        covered_total += covered.sum().item()
    return covered_total / max(valid_total, 1)

coverage = compute_baseline_coverage(baseline_model, city_loader, ADE_TO_OURS)
print(f"\nBaseline coverage on valid GT pixels: {coverage:.2%}")
print("(Fraction of GT pixels where baseline predicted one of our 4 classes.")
print(" If low, baseline is predicting many ADE classes that don't map to ours.)")

In [ ]:
# === CELL 24: Save baseline results + comparison table ===
import json
with open(f"{CKPT_DIR}/baseline_results.json", "w") as f:
    json.dump({
        "miou": baseline_results["miou"],
        "per_class_iou": baseline_results["per_class_iou"],
        "per_class_acc": baseline_results["per_class_acc"],
        "coverage": coverage,
        "class_names": CLASS_NAMES,
    }, f, indent=2)

# Side-by-side comparison
print("\n" + "=" * 60)
print(f"{'class':<12} {'baseline':>12} {'ours':>12} {'delta':>10}")
print("-" * 60)
for i, name in enumerate(CLASS_NAMES):
    b = baseline_results["per_class_iou"][i]
    o = city_results["per_class_iou"][i]
    print(f"{name:<12} {b:>12.4f} {o:>12.4f} {o-b:>+10.4f}")
print("-" * 60)
print(f"{'mIoU':<12} {baseline_results['miou']:>12.4f} {city_results['miou']:>12.4f} "
      f"{city_results['miou']-baseline_results['miou']:>+10.4f}")
print("=" * 60)